# 02 - Modeling: Perbandingan Model Klasifikasi "Termurah" (CPU only)

Notebook ini melanjutkan hasil `01-preprocessing` (train.csv / test.csv) untuk melatih dan **membandingkan 5 model klasifikasi klasik yang murah secara komputasi**:

1. **Logistic Regression**
2. **Decision Tree**
3. **Gaussian Naive Bayes**
4. **K-Nearest Neighbors (KNN)**
5. **Random Forest** (ensemble ringan, sebagai pembanding)

Semua model dipilih karena **ringan dijalankan di CPU** (tidak butuh GPU sama sekali) — cocok dengan target akselerator notebook ini: **CPU / None, bukan GPU T4**.

Perbandingan dilakukan dari dua sisi:
- **"Kemurahan" (cost)**: waktu training & waktu prediksi (dari `cross_validate` dan holdout test).
- **Performa**: accuracy, precision, recall, F1-score, ROC-AUC (dataset target imbalanced ~85/15, jadi accuracy saja tidak cukup).

In [ ]:
import os
import glob
import time
import json as _json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 0. Memuat Hasil Preprocessing (`train.csv`, `test.csv`)

Notebook ini mengharapkan output dari `01-preprocessing` sudah ditambahkan sebagai *data source* (Add Data → Notebook Output) di Kaggle. Jika belum ditemukan, fallback ke pencarian otomatis di `/kaggle/input/**/train.csv`.

In [ ]:
def find_file(filename, extra_candidates=None):
    candidates = list(extra_candidates or [])
    candidates += [
        f"/kaggle/input/online-shoppers-01-preprocessing/{filename}",
        f"/kaggle/working/{filename}",
        f"../01-preprocessing/{filename}",
        filename,
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    # fallback: cari di mana pun di bawah /kaggle/input
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if matches:
        return matches[0]
    return None


train_path = find_file("train.csv")
test_path = find_file("test.csv")

if train_path is None or test_path is None:
    raise FileNotFoundError(
        "train.csv/test.csv tidak ditemukan. Jalankan notebook '01-preprocessing' terlebih dahulu, "
        "lalu tambahkan output-nya sebagai Data Source pada notebook ini (Add Data > Notebook Output)."
    )

print("train:", train_path)
print("test :", test_path)

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X_train = train_df.drop(columns=["Revenue"])
y_train = train_df["Revenue"]
X_test = test_df.drop(columns=["Revenue"])
y_test = test_df["Revenue"]

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Distribusi target train:\n", y_train.value_counts(normalize=True))


## 1. Definisi 5 Model "Murah" (CPU only)

Semua model dibungkus `Pipeline` dengan `StandardScaler` (fit hanya di data train tiap fold, tidak ada data leakage). `n_jobs=1` dipakai secara konsisten supaya perbandingan waktu training antar model adil (single-core, bukan dibantu paralelisme).

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Decision Tree": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", DecisionTreeClassifier(max_depth=10, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Naive Bayes (Gaussian)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", GaussianNB()),
    ]),
    "K-Nearest Neighbors": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=15, n_jobs=1)),
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(
            n_estimators=100, max_depth=10, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=1,
        )),
    ]),
}

list(models.keys())


## 2. Cross-Validation: Perbandingan Cost (waktu) & Performa

Menggunakan `StratifiedKFold` 5-fold pada data train. `cross_validate` otomatis mencatat `fit_time` dan `score_time` per fold — ini dipakai sebagai ukuran "kemurahan" komputasi tiap model.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

cv_rows = []
cv_raw = {}

for name, pipe in models.items():
    print(f"Menjalankan 5-fold CV untuk: {name} ...")
    result = cross_validate(
        pipe, X_train, y_train, cv=cv, scoring=scoring,
        n_jobs=1, return_train_score=False,
    )
    cv_raw[name] = result
    cv_rows.append({
        "model": name,
        "fit_time_sec_mean": result["fit_time"].mean(),
        "fit_time_sec_std": result["fit_time"].std(),
        "score_time_sec_mean": result["score_time"].mean(),
        "accuracy_mean": result["test_accuracy"].mean(),
        "precision_mean": result["test_precision"].mean(),
        "recall_mean": result["test_recall"].mean(),
        "f1_mean": result["test_f1"].mean(),
        "roc_auc_mean": result["test_roc_auc"].mean(),
    })

cv_summary = pd.DataFrame(cv_rows).sort_values("fit_time_sec_mean").reset_index(drop=True)
cv_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

order = cv_summary.sort_values("fit_time_sec_mean")
axes[0].barh(order["model"], order["fit_time_sec_mean"], color="#4C72B0")
axes[0].set_xlabel("Rata-rata waktu training per fold (detik)")
axes[0].set_title("Cost: Waktu Training (5-fold CV)")
axes[0].invert_yaxis()

order2 = cv_summary.sort_values("f1_mean", ascending=False)
axes[1].barh(order2["model"], order2["f1_mean"], color="#55A868")
axes[1].set_xlabel("Rata-rata F1-score (5-fold CV)")
axes[1].set_title("Performa: F1-score (5-fold CV)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


## 3. Evaluasi Akhir di Test Set (Holdout)

Melatih ulang tiap model pada **seluruh** data train, lalu diuji pada data test yang belum pernah dilihat. Waktu training penuh dan waktu prediksi juga dicatat sebagai ukuran cost tambahan.

In [ ]:
test_rows = []
fitted_models = {}
roc_data = {}

for name, pipe in models.items():
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred = pipe.predict(X_test)
    predict_time = time.perf_counter() - t0

    y_proba = pipe.predict_proba(X_test)[:, 1]

    fitted_models[name] = pipe
    roc_data[name] = roc_curve(y_test, y_proba)

    test_rows.append({
        "model": name,
        "fit_time_full_train_sec": fit_time,
        "predict_time_test_sec": predict_time,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    })

test_summary = pd.DataFrame(test_rows).sort_values("f1", ascending=False).reset_index(drop=True)
test_summary


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for i, (name, pipe) in enumerate(fitted_models.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    ax = axes[i]
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["No", "Yes"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["No", "Yes"])
    for r in range(2):
        for c in range(2):
            ax.text(c, r, cm[r, c], ha="center", va="center", color="black")

for j in range(len(fitted_models), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6.5, 6))
for name, (fpr, tpr, _) in roc_data.items():
    auc = test_summary.loc[test_summary["model"] == name, "roc_auc"].values[0]
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Perbandingan 5 Model")
plt.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()


## 4. Ringkasan Perbandingan: Cost vs Performa

In [ ]:
test_summary_renamed = test_summary[[
    "model", "fit_time_full_train_sec", "predict_time_test_sec", "accuracy", "f1", "roc_auc"
]].rename(columns={"accuracy": "accuracy_test", "f1": "f1_test", "roc_auc": "roc_auc_test"})

summary = cv_summary.merge(test_summary_renamed, on="model")
summary = summary.sort_values("fit_time_sec_mean")
summary


In [ ]:
cheapest = summary.sort_values("fit_time_sec_mean").iloc[0]
best_f1 = summary.sort_values("f1_test", ascending=False).iloc[0]
best_auc = summary.sort_values("roc_auc_test", ascending=False).iloc[0]

print(f"Model TERMURAH (waktu training tercepat): {cheapest['model']} "
      f"({cheapest['fit_time_sec_mean']*1000:.2f} ms/fold)")
print(f"Model dengan F1-score TERBAIK di test set : {best_f1['model']} (F1={best_f1['f1_test']:.4f})")
print(f"Model dengan ROC-AUC TERBAIK di test set   : {best_auc['model']} (AUC={best_auc['roc_auc_test']:.4f})")


**Interpretasi:**
- Kolom `fit_time_sec_mean` (dari 5-fold CV) dan `fit_time_full_train_sec` (full train) adalah ukuran **kemurahan komputasi**: makin kecil, makin murah/cepat model dilatih.
- Naive Bayes dan Logistic Regression biasanya menjadi model termurah karena tidak melakukan pencarian split (Decision Tree/Random Forest) atau penyimpanan seluruh data (KNN).
- Random Forest umumnya paling mahal di antara kelimanya (karena melatih banyak pohon), tapi sering memberi performa (F1/ROC-AUC) terbaik — ini trade-off klasik *cost vs performance*.
- Untuk kebutuhan yang mengutamakan efisiensi (mis. inferensi real-time, sumber daya terbatas, dataset besar), model termurah dengan performa yang masih kompetitif adalah pilihan yang disarankan.

In [ ]:
out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
out_path = os.path.join(out_dir, "model_comparison_results.csv")
summary.to_csv(out_path, index=False)
print("Hasil perbandingan disimpan ke:", out_path)


## Kesimpulan

Notebook ini membandingkan 5 model klasifikasi yang murah secara komputasi (Logistic Regression, Decision Tree, Naive Bayes, KNN, Random Forest) untuk memprediksi `Revenue` (purchase intention) pada dataset Online Shoppers Purchasing Intention, dijalankan sepenuhnya di **CPU** tanpa GPU.

Lihat tabel `summary` di atas dan file output `model_comparison_results.csv` untuk detail lengkap trade-off cost (waktu training/prediksi) vs performa (accuracy, precision, recall, F1, ROC-AUC) tiap model.